# Resilient Rover v0 — Environment Prototype

This notebook prototypes the first Resilient Rover reinforcement-learning environment.

The purpose of v0 is to create a simple one-dimensional environment that makes the mechanics of Gymnasium and Q-learning easy to inspect.

The rover must reach a target position before exhausting its battery.

## Environment Design

### World

The environment is a one-dimensional world with discrete positions:

0 -- 1 -- 2 -- 3 -- 4 -- 5 -- 6 -- 7 -- 8 -- 9

### Observation

The agent observes:

- rover position
- target position
- remaining battery

### Actions

- 0 = wait
- 1 = move right
- 2 = move left

### Rewards

- +100 for reaching the target
- -100 for exhausting the battery before reaching the target
- -1 for every other timestep

### Episode End Conditions

The episode terminates when:

- the rover reaches the target, or
- the battery reaches zero

The episode is truncated when the maximum number of steps is reached.

## Imports

In [3]:
# from typing import Optional

# import gymnasium as gym
# import numpy as np

## Environment Configuration

These constants define the initial v0 environment parameters.

In [4]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [5]:
import gymnasium as gym

from resilient_rover.envs.line_world_env import (
    LineWorldEnv,
    WORLD_SIZE,
    MAX_BATTERY,
    MAX_STEPS,
)

## ResilientRoverEnv

The environment follows the Gymnasium `Env` interface.

Internal state:

- rover position
- target position
- battery level
- current step count

Register the environment so we can create it with gym.make()

In [6]:
gym.register(
    id="resilient_rover/LineWorld-v0",
    entry_point=LineWorldEnv,
)

In [7]:
env = gym.make("resilient_rover/LineWorld-v0")
obs, info = env.reset()

print(obs)
print(info)

{'rover': 6, 'target': 0, 'battery': 20}
{'distance_to_target': 6, 'battery_level': 20, 'step_count': 0}


In [8]:
action = 1  # Move right

observation, reward, terminated, truncated, info = env.step(action)

print(observation)
print(reward)
print(terminated)
print(truncated)
print(info)

{'rover': 7, 'target': 0, 'battery': 19}
-1
False
False
{'distance_to_target': 7, 'battery_level': 19, 'step_count': 1}


## Environment Validation

In [9]:
env = gym.make("resilient_rover/LineWorld-v0")

observation, info = env.reset(seed=42)

print(observation)
print(info)

assert 0 <= observation["rover"] < WORLD_SIZE
assert 0 <= observation["target"] < WORLD_SIZE
assert observation["rover"] != observation["target"]
assert observation["battery"] == MAX_BATTERY
assert info["step_count"] == 0

print("Reset validation passed.")

{'rover': 0, 'target': 7, 'battery': 20}
{'distance_to_target': 7, 'battery_level': 20, 'step_count': 0}
Reset validation passed.


In [10]:
env = LineWorldEnv()
env.reset(seed=42)

env._rover_location = 4
env._target_location = 8
env._battery_level = 20
env._step_count = 0

observation, reward, terminated, truncated, info = env.step(0)

assert observation["rover"] == 4
assert observation["battery"] == 19
assert info["step_count"] == 1
assert reward == -1
assert terminated is False
assert truncated is False

In [11]:
env = LineWorldEnv()
env.reset(seed=42)

env._rover_location = 4
env._target_location = 8
env._battery_level = 20
env._step_count = 0

observation, reward, terminated, truncated, info = env.step(1)

assert observation["rover"] == 5
assert observation["battery"] == 19
assert info["step_count"] == 1
assert reward == -1
assert terminated is False
assert truncated is False

In [12]:
env = LineWorldEnv()
env.reset(seed=42)

env._rover_location = 4
env._target_location = 8
env._battery_level = 20
env._step_count = 0

observation, reward, terminated, truncated, info = env.step(2)

assert observation["rover"] == 3
assert observation["battery"] == 19
assert info["step_count"] == 1
assert reward == -1
assert terminated is False
assert truncated is False

In [13]:
env = LineWorldEnv()
env.reset(seed=42)

env._rover_location = 0
env._target_location = 8
env._battery_level = 20
env._step_count = 0

observation, reward, terminated, truncated, info = env.step(2)

assert observation["rover"] == 0
assert observation["battery"] == 19
assert info["step_count"] == 1
assert reward == -1
assert terminated is False
assert truncated is False

In [14]:
env = LineWorldEnv()
env.reset(seed=42)

env._rover_location = WORLD_SIZE - 1
env._target_location = 4
env._battery_level = 20
env._step_count = 0

observation, reward, terminated, truncated, info = env.step(1)

assert observation["rover"] == WORLD_SIZE - 1
assert observation["battery"] == 19
assert info["step_count"] == 1
assert reward == -1
assert terminated is False
assert truncated is False

In [15]:
env = LineWorldEnv()
env.reset(seed=42)

env._rover_location = 3
env._target_location = 4
env._battery_level = 20
env._step_count = 0

observation, reward, terminated, truncated, info = env.step(1)

assert observation["rover"] == 4
assert observation["battery"] == 19
assert info["step_count"] == 1
assert reward == 100
assert terminated is True
assert truncated is False

In [16]:
env = LineWorldEnv()
env.reset(seed=42)

env._rover_location = 4
env._target_location = 8
env._battery_level = 1
env._step_count = 0

observation, reward, terminated, truncated, info = env.step(0)

assert observation["rover"] == 4
assert observation["battery"] == 0
assert info["step_count"] == 1
assert reward == -100
assert terminated is True
assert truncated is False

In [17]:
env = LineWorldEnv()
env.reset(seed=42)

env._rover_location = 4
env._target_location = 8
env._battery_level = 20
env._step_count = MAX_STEPS - 1

observation, reward, terminated, truncated, info = env.step(1)

assert observation["rover"] == 5
assert observation["battery"] == 19
assert info["step_count"] == MAX_STEPS
assert reward == -1
assert terminated is False
assert truncated is True

In [18]:
env = LineWorldEnv()
env.reset(seed=42)

env._rover_location = 7
env._target_location = 8
env._battery_level = 1
env._step_count = 0

observation, reward, terminated, truncated, info = env.step(1)

assert observation["rover"] == 8
assert observation["battery"] == 0
assert info["step_count"] == 1
assert reward == 100
assert terminated is True
assert truncated is False

In [19]:
env = LineWorldEnv()

obs1, info1 = env.reset(seed=42)
obs2, info2 = env.reset(seed=42)

assert obs1 == obs2
assert info1 == info2

print("Seeded reset reproducibility test passed.")

Seeded reset reproducibility test passed.


In [20]:
from gymnasium.utils.env_checker import check_env

env = LineWorldEnv()

try:
    check_env(env)
    print("Environment passes all checks!")
except Exception as e:
    print(f"Environment has issues: {e}")

Environment passes all checks!


/home/ryne/Projects/personal/resilient-rover/.venv/lib/python3.12/site-packages/gymnasium/utils/env_checker.py:440: UserWarning: WARN: Not able to test alternative render modes due to the environment not having a spec. Try instantiating the environment through `gymnasium.make`
  logger.warn(
